# RL-дообучение GPT-2 рекомендера на KION

План: данные и разбиение по времени → базовая GPT-2 → reward model → RL (REINFORCE/GRPO + KL) → анализ.
Статус: каркас, секции заполняются по мере работы.

In [ ]:
import os, glob, json, random
from pathlib import Path
import numpy as np

ON_KAGGLE = os.path.exists("/kaggle/input")
SMOKE = True          # True: малая выборка, быстрый прогон; False: полный прогон на GPU
SEED = 42

WORK = Path("/kaggle/working") if ON_KAGGLE else Path(".")
CACHE, CKPT, RESULTS = WORK/"cache", WORK/"checkpoints", WORK/"results"
for d in (CACHE, CKPT, RESULTS): d.mkdir(exist_ok=True)

def find_data_dir():
    if os.environ.get("DATA_DIR"): return Path(os.environ["DATA_DIR"])
    hits = glob.glob("/kaggle/input/**/interactions.csv", recursive=True)
    if not hits: raise FileNotFoundError("interactions.csv не найден: подключите датасет или задайте DATA_DIR")
    return Path(hits[0]).parent
DATA_DIR = find_data_dir()

random.seed(SEED); np.random.seed(SEED)
print("kaggle:", ON_KAGGLE, "| smoke:", SMOKE, "| data:", DATA_DIR)

## 1. Данные и разбиение по времени

**Протокол разбиения** (один для всех моделей):

- Глобальные даты: train `< 2021-08-02`, val `2021-08-02 … 08-08`, test `2021-08-09 … 08-22` (последние 14 дней).
- Обучение (GPT-2, reward model, RL) использует только train. Val нужна для подбора настроек, test открываем один раз в конце.
- Eval-кейс: пользователь с ≥5 событиями в train и ≥1 событием в периоде. Цель — первое событие в периоде, контекст — все события пользователя до начала периода (для test это train+val). Следующие 5 событий периода сохраняем как «будущее» для проверки 5-шаговой генерации.
- Объекты, которых нет в train, из val/test выкидываем (холодный старт здесь не исследуем), долю считаем и печатаем.
- В KION есть только дата без времени: порядок внутри дня берём из порядка строк файла (стабильная сортировка). Это шум, он описан в ограничениях.
- Пара (пользователь, объект) в данных не повторяется, поэтому уже просмотренные объекты исключаем из рекомендаций при оценке.

In [ ]:
import pandas as pd

TRAIN_END = pd.Timestamp("2021-08-02")   # train: dt < TRAIN_END
VAL_END   = pd.Timestamp("2021-08-09")   # val: [TRAIN_END, VAL_END), test: >= VAL_END
MIN_HIST  = 5                            # минимум событий в train
N_FUTURE  = 5                            # длина «будущего» для многошагового анализа
SMOKE_USERS = 30_000                     # размер выборки пользователей при SMOKE=True
TAG = "smoke" if SMOKE else "full"

In [ ]:
def build_split():
    df = pd.read_csv(DATA_DIR/"interactions.csv", parse_dates=["last_watch_dt"])
    n_raw = len(df)
    df = df.dropna(subset=["watched_pct"]).reset_index(drop=True)
    df["row"] = np.arange(len(df))
    df = df.sort_values(["user_id", "last_watch_dt", "row"], kind="mergesort").reset_index(drop=True)
    df["period"] = np.select([df.last_watch_dt < TRAIN_END, df.last_watch_dt < VAL_END], [0, 1], 2)  # 0 train, 1 val, 2 test

    n_train = df[df.period == 0].groupby("user_id").size()
    keep = n_train[n_train >= MIN_HIST].index
    if SMOKE:
        keep = pd.Index(np.random.RandomState(SEED).choice(keep, SMOKE_USERS, replace=False))
    df = df[df.user_id.isin(keep)].copy()

    # словарь объектов только по train; 0 зарезервирован под PAD
    train_items = np.sort(df.loc[df.period == 0, "item_id"].unique())
    item2idx = {it: i + 1 for i, it in enumerate(train_items)}
    seen = df.item_id.isin(item2idx)
    unseen_share = {p: float((~seen[df.period == p]).mean()) for p in (1, 2)}
    df = df[seen].copy()
    df["item"] = df.item_id.map(item2idx).astype("int32")
    df = df[["user_id", "item", "item_id", "last_watch_dt", "watched_pct", "total_dur", "period"]].reset_index(drop=True)
    return df, item2idx, unseen_share, n_raw

cache_f = CACHE/f"interactions_{TAG}.parquet"
if cache_f.exists():
    df = pd.read_parquet(cache_f)
    item2idx = {it: i + 1 for i, it in enumerate(np.sort(df.loc[df.period == 0, "item_id"].unique()))}
    unseen_share = json.load(open(RESULTS/f"split_stats_{TAG}.json"))["unseen_share"]
else:
    df, item2idx, unseen_share, n_raw = build_split()
    df.to_parquet(cache_f)
n_items = len(item2idx) + 1   # с учётом PAD=0
print(df.shape, "users:", df.user_id.nunique(), "| vocab:", n_items)

Eval-кейсы: цель, контекст (индексы строк в `df`) и «будущее» из 5 событий.

In [ ]:
def build_eval(df, period):
    """Одна строка на пользователя: первое событие периода — цель, следующие N_FUTURE — будущее."""
    cur = df[df.period == period]
    g = cur.groupby("user_id", sort=False)
    first = g.head(1).set_index("user_id")
    fut = g.head(1 + N_FUTURE).groupby("user_id", sort=False).item.apply(lambda s: s.iloc[1:].tolist())
    out = pd.DataFrame({"target": first["item"], "target_pct": first["watched_pct"],
                        "target_dt": first["last_watch_dt"], "future": fut})
    return out

evals = {"val": build_eval(df, 1), "test": build_eval(df, 2)}

def history(user_id, period):
    """Контекст пользователя: все события строго до начала периода."""
    u = df_by_user.get_group(user_id)
    return u[u.period < period]

df_by_user = df.groupby("user_id", sort=False)
for k, e in evals.items(): print(k, "eval users:", len(e))

Проверки на утечки и статистика.

In [ ]:
tr, va, te = (df[df.period == p] for p in (0, 1, 2))
assert tr.last_watch_dt.max() < TRAIN_END
assert va.last_watch_dt.between(TRAIN_END, VAL_END - pd.Timedelta(days=1)).all()
assert te.last_watch_dt.min() >= VAL_END
assert df.groupby("user_id").last_watch_dt.apply(lambda s: s.is_monotonic_increasing).all()   # сортировка по времени
assert set(va.item) <= set(tr.item) and set(te.item) <= set(tr.item)                             # словарь только из train
assert not df.duplicated(["user_id", "item"]).any()                                              # нет повторов, значит цель не из истории
for k, p in (("val", 1), ("test", 2)):                                                           # цель всегда позже истории
    e = evals[k]; hist_last = df[df.period < p].groupby("user_id").last_watch_dt.max()
    assert (e.target_dt.values >= hist_last.reindex(e.index).values).all()

stats = {
    "tag": TAG, "users": int(df.user_id.nunique()), "vocab": int(n_items),
    "events": {"train": len(tr), "val": len(va), "test": len(te)},
    "train_seq_len_median": float(tr.groupby("user_id").size().median()),
    "eval_users": {k: len(e) for k, e in evals.items()},
    "unseen_share": unseen_share,
    "watched_pct_mean": {"train": float(tr.watched_pct.mean()), "val": float(va.watched_pct.mean()), "test": float(te.watched_pct.mean())},
    "share_pct_ge_50": {"train": float((tr.watched_pct >= 50).mean()), "test": float((te.watched_pct >= 50).mean())},
}
json.dump(stats, open(RESULTS/f"split_stats_{TAG}.json", "w"), indent=1)
print(json.dumps(stats, indent=1)); print("утечек не найдено")

## 2. Базовая модель (GPT-2, next item prediction)

## 3. Reward model (watched_pct)

## 4. RL-дообучение (REINFORCE/GRPO + KL)

## 5. Анализ: reward vs релевантность, разнообразие, reward hacking

## 6. Генерация на 5 шагов вперёд

## 7. Варианты функции вознаграждения и компромисс

## 8. Выводы и ограничения